# Phase 2 — Serving baseline + single-optimization marginals

Runs EXPERIMENT_MATRIX.md blocks 1–2 (IMPLEMENTATION_PLAN.md Phase 2) on the anchor model
**Llama-3.1-8B-Instruct**: the FP16/no-opt serving baseline plus each optimization ALONE —
W (AWQ W4A16), K (FP8 KV-cache, software-emulated on A100 — a documented deployment
condition, EXPERIMENT_MATRIX §7), S (EAGLE-3) — across GSM8K, HumanEval, and the RAG
shared-prefix workload at concurrency {1, 8, 32, 64}.

**48 run cells, 4 server launches.** Every run records goodput (verified-and-generated
tokens/sec) and the **measured emergent batch size** — concurrency is set, batch size is
never set (PROJECT_SPEC §7.2).

**Requirements**
- **A100 runtime** — verify with the cell below, don't trust the dropdown.
- **HF token** as Colab secret `HF_TOKEN`, with the **Llama-3.1** license accepted
  (separate gate from Llama-3: visit meta-llama/Llama-3.1-8B-Instruct and accept).

**Budget:** ~2.5–3.5 A100-hours ≈ 30–45 units at the calibrated ~12 units/hr
(PREREQ_RESULTS Check 1). Fully resumable: a disconnect costs one cell — re-run the
sweep cell and completed cells are skipped. Note the units meter before/after.

In [ ]:
# 1. Verify the GPU actually assigned
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
units_before = input("Compute-units balance right now: ")

In [ ]:
# 2. Get the repo + Spec-Bench (RAG passages come from its 'rag' subtask;
#    external/ is git-ignored so it must be cloned per-session)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK" 

In [ ]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)" 

In [ ]:
# 4. HF auth (gated Llama-3.1 weights)
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 5. Harness self-check (no GPU needed, ~1 min). Includes the RAG
#    byte-identical-prefix unit tests. Fix anything here before burning GPU time.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

In [ ]:
# 6. Shared-prefix token-ID check with the REAL Llama-3.1 tokenizer
#    (HARNESS_SPEC §7/§10 guard, beyond the synthetic-tokenizer unit test)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python scripts/check_rag_prefix.py

In [ ]:
# 7. Sanity: the four server commands (nothing launches)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_base_*_r0.yaml" "configs/factorial/cube_w_*_r0.yaml" "configs/factorial/cube_k_*_r0.yaml" "configs/factorial/cube_s_*_r0.yaml" --results-dir results --dry-run 2>&1 | grep "server command" 

In [ ]:
# 8. THE SWEEP: 48 cells, 4 server groups, resumable. ~2.5-3.5 h.
# Order: baseline group first, then K (fp8 kv), then S (eagle3), then W (AWQ).
# If Colab disconnects: re-run this cell; completed cells are skipped.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_base_*_r0.yaml" "configs/factorial/cube_w_*_r0.yaml" "configs/factorial/cube_k_*_r0.yaml" "configs/factorial/cube_s_*_r0.yaml" --results-dir results

In [ ]:
# 9. Marginals report: goodput vs concurrency per optimization, with
#    speedup vs baseline, emergent batch size, and tau for the EAGLE-3 cells
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.marginals results

In [ ]:
# 10. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase2_results.zip results
from google.colab import files
files.download("phase2_results.zip")

## Reading the result

The marginals table is the Phase-2 deliverable. What to look for:

- **S (EAGLE-3) column:** relative speedup should be largest at conc=1 and erode as
  concurrency grows (the serving-regime thesis). Find the crossover — the concurrency
  where it drops below ~1.0x — per workload; that number is hypothesis "S main"
  (EXPERIMENT_MATRIX §5) answered.
- **W (AWQ) column:** strong win at conc=1 (memory-bound decode); watch whether it
  shrinks or flips at conc 32–64 as the regime turns compute-bound — dequant overhead
  is the same mechanism as the Block-0 finding, now on the serving axis.
- **K (FP8-KV) column:** expect ≈neutral speed on short prompts (GSM8K/HumanEval) —
  it's emulated on A100 — but a real gain on RAG at high concurrency via *capacity*:
  compare the **emergent batch** columns; FP8-KV should sustain a larger running batch
  before preemption.
- **Emergent batch vs concurrency 64 on RAG:** expect batch ≪ 64 (KV capacity limits
  it) — that gap is the capacity mechanism made visible, and it's the reason batch
  size must be measured, not set.
- **tau across concurrency:** should be roughly stable — if it collapses at high
  concurrency, inspect before interpreting (scheduler behavior, not acceptance).

Afterwards: commit `results/` runs + the report (or the zip), update PREREQ_RESULTS
Check 1 with this session's burn rate, and Phase 3 is just the remaining four corners
of the cube through the same sweep machinery + `analysis/factorial.py`.